# 📈 Notebook 05 — Business Impact & A/B Test Simulation

This notebook simulates an **A/B test** comparing four recommendation strategies:

| Strategy | Description |
|----------|-------------|
| Random Baseline | Random add-on from same cuisine |
| Apriori Generic | Classic association rules (Apriori) |
| SmartCart v1 (ML) | ML conversion model only |
| **SmartCart v2 (Full)** | **Full 6-layer pipeline** |

## Metrics
- **ATC Rate** — Add-to-Cart click rate
- **AOV Uplift** — Average Order Value increase from add-ons
- **KPT Impact** — Average delay added to orders
- **Diversity** — Avg unique categories per recommendation rail
- **₹ Revenue Uplift** — Daily / Monthly business impact

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Setup complete')

In [ ]:
# A/B test simulation
N_ORDERS = 10_000  # simulate 10,000 orders

def simulate_strategy(n, atc_base, aov_mean, aov_std, kpt_impact_mean, diversity_mean,
                       noise=0.02, name='Strategy'):
    """Simulate orders for a recommendation strategy."""
    clicks = np.random.binomial(1, atc_base, n)
    aov_uplift = np.where(clicks == 1,
                          np.random.normal(aov_mean, aov_std, n).clip(10, 500),
                          0)
    kpt_impact = np.where(clicks == 1,
                          np.random.exponential(kpt_impact_mean, n).clip(0, 30),
                          0)
    diversity = np.random.normal(diversity_mean, 0.3, n).clip(1, 4)

    return pd.DataFrame({
        'strategy': name,
        'clicked': clicks,
        'aov_uplift': aov_uplift,
        'kpt_impact': kpt_impact,
        'diversity': diversity,
    })

# Simulate each strategy
df_random    = simulate_strategy(N_ORDERS, 0.15, 20,  8,  5.0, 1.5, name='Random Baseline')
df_apriori   = simulate_strategy(N_ORDERS, 0.25, 35,  12, 3.0, 1.8, name='Apriori Generic')
df_sc_v1     = simulate_strategy(N_ORDERS, 0.32, 48,  15, 2.0, 2.0, name='SmartCart v1 (ML)')
df_sc_v2     = simulate_strategy(N_ORDERS, 0.38, 62,  18, 0.1, 2.3, name='SmartCart v2 (Full)')

df_all = pd.concat([df_random, df_apriori, df_sc_v1, df_sc_v2], ignore_index=True)

# Aggregate metrics
summary = df_all.groupby('strategy').agg(
    ATC_Rate=('clicked', 'mean'),
    AOV_Uplift=('aov_uplift', lambda x: x[x > 0].mean()),
    KPT_Impact=('kpt_impact', lambda x: x[x > 0].mean()),
    Diversity=('diversity', 'mean'),
).round(2)

order = ['Random Baseline', 'Apriori Generic', 'SmartCart v1 (ML)', 'SmartCart v2 (Full)']
summary = summary.reindex(order)
summary['ATC_Rate_pct'] = (summary['ATC_Rate'] * 100).round(1)

print('A/B Test Simulation Results:')
print(summary[['ATC_Rate_pct', 'AOV_Uplift', 'KPT_Impact', 'Diversity']].to_string())

In [ ]:
# Metrics comparison bar charts
strategies = summary.index.tolist()
colors = ['#95A5A6', '#F39C12', '#3498DB', '#E74C3C']

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

def make_bar(ax, values, title, ylabel, colors, highlight_last=True, fmt='{:.1f}'):
    bar_colors = [c if (not highlight_last or i < len(colors)-1) else '#E74C3C' for i, c in enumerate(colors)]
    bars = ax.bar(strategies, values, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
                fmt.format(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

make_bar(fig.add_subplot(gs[0, 0]), summary['ATC_Rate_pct'].values,
         'Add-to-Cart Rate (%)', 'ATC Rate (%)', colors, fmt='{:.1f}%')

make_bar(fig.add_subplot(gs[0, 1]), summary['AOV_Uplift'].values,
         'AOV Uplift (₹ per clicked order)', 'AOV Uplift (₹)', colors, fmt='₹{:.0f}')

ax3 = fig.add_subplot(gs[1, 0])
kpt_colors = ['#E74C3C' if v > 1 else '#2ECC71' for v in summary['KPT_Impact'].values]
kpt_colors[-1] = '#2ECC71'  # SmartCart v2 is best
bars = ax3.bar(strategies, summary['KPT_Impact'].values, color=kpt_colors, edgecolor='white')
ax3.set_title('Avg KPT Impact (min delay added)', fontsize=12, fontweight='bold', pad=8)
ax3.set_ylabel('KPT Delay (min)')
ax3.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, summary['KPT_Impact'].values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.1f}min', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax3.axhline(0, color='black', linewidth=0.5)

make_bar(fig.add_subplot(gs[1, 1]), summary['Diversity'].values,
         'Rail Diversity (avg unique categories)', 'Unique Categories', colors, fmt='{:.1f}')

fig.suptitle('A/B Test: Recommendation Strategy Comparison', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('ab_test_metrics.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved as ab_test_metrics.png')

In [ ]:
# Business impact projection
DAILY_ORDERS   = 500_000   # Zomato scale ~500K orders/day
AVG_CART       = 280       # Average cart value ₹280

projections = []
for strat in order:
    row = summary.loc[strat]
    atc_rate   = row['ATC_Rate']
    aov_uplift = row['AOV_Uplift']

    orders_with_addon = DAILY_ORDERS * atc_rate
    daily_revenue_uplift  = orders_with_addon * aov_uplift
    monthly_revenue_uplift = daily_revenue_uplift * 30
    annual_revenue_uplift  = daily_revenue_uplift * 365

    projections.append({
        'Strategy': strat,
        'Daily Orders w/ Add-on': f'{orders_with_addon:,.0f}',
        'Daily Revenue Uplift':   f'₹{daily_revenue_uplift/1e5:.1f}L',
        'Monthly Revenue Uplift': f'₹{monthly_revenue_uplift/1e7:.1f}Cr',
        'Annual Revenue Uplift':  f'₹{annual_revenue_uplift/1e7:.0f}Cr',
    })

df_proj = pd.DataFrame(projections)
print('\n💰 Business Impact Projection (@ 500K daily orders):')
print(df_proj.to_string(index=False))

# Incremental uplift vs baseline
baseline_daily = DAILY_ORDERS * summary.loc['Random Baseline', 'ATC_Rate'] * summary.loc['Random Baseline', 'AOV_Uplift']
sc_v2_daily    = DAILY_ORDERS * summary.loc['SmartCart v2 (Full)', 'ATC_Rate'] * summary.loc['SmartCart v2 (Full)', 'AOV_Uplift']
incremental_monthly = (sc_v2_daily - baseline_daily) * 30
print(f'\n📊 Incremental uplift (SmartCart v2 vs Random Baseline):')
print(f'   Daily:   ₹{(sc_v2_daily - baseline_daily)/1e5:.1f}L additional revenue')
print(f'   Monthly: ₹{incremental_monthly/1e7:.1f}Cr additional revenue')

In [ ]:
# Revenue uplift chart
daily_uplifts = []
for strat in order:
    row = summary.loc[strat]
    daily_uplifts.append(DAILY_ORDERS * row['ATC_Rate'] * row['AOV_Uplift'] / 1e5)  # in Lakhs

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Daily revenue uplift
bar_colors_proj = ['#95A5A6', '#F39C12', '#3498DB', '#E74C3C']
bars = axes[0].bar(order, daily_uplifts, color=bar_colors_proj, edgecolor='white')
axes[0].set_title('Daily Revenue Uplift (₹ Lakhs)\n@ 500K orders/day', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Daily Revenue Uplift (₹L)')
axes[0].tick_params(axis='x', rotation=25)
for bar, val in zip(bars, daily_uplifts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'₹{val:.0f}L', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Incremental uplift over baseline
baseline_val = daily_uplifts[0]
incremental = [v - baseline_val for v in daily_uplifts]
inc_colors = ['#BDC3C7' if i == 0 else '#2ECC71' for i in range(4)]
bars2 = axes[1].bar(order, incremental, color=inc_colors, edgecolor='white')
axes[1].set_title('Incremental Daily Revenue\nvs Random Baseline (₹ Lakhs)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Incremental Revenue (₹L/day)')
axes[1].tick_params(axis='x', rotation=25)
axes[1].axhline(0, color='gray', linewidth=0.8, linestyle='--')
for bar, val in zip(bars2, incremental):
    if val > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.3,
                     f'+₹{val:.0f}L', ha='center', va='bottom', fontsize=11, fontweight='bold', color='#27AE60')

plt.tight_layout()
plt.savefig('business_impact.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved as business_impact.png')

In [ ]:
# Final summary table
print('=' * 75)
print('SMARTCART AI — FINAL BUSINESS IMPACT SUMMARY')
print('=' * 75)
print(f'\n{"Strategy":<25} {"ATC%":>6} {"AOV Uplift":>12} {"KPT Impact":>12} {"Diversity":>10}')
print('-' * 75)
emojis = ['⚪', '🟡', '🔵', '🔴']
for i, strat in enumerate(order):
    row = summary.loc[strat]
    print(f"{emojis[i]} {strat:<23} {row['ATC_Rate_pct']:>5.1f}%"
          f"  {'₹'+str(int(row['AOV_Uplift'])):>10}"
          f"  {str(round(row['KPT_Impact'],1))+'min':>10}"
          f"  {row['Diversity']:>9.1f}")

print('-' * 75)
v2 = summary.loc['SmartCart v2 (Full)']
rb = summary.loc['Random Baseline']
print(f"\n🚀 SmartCart v2 vs Random Baseline:")
print(f"   ATC Rate: +{(v2['ATC_Rate']/rb['ATC_Rate'] - 1)*100:.0f}% relative improvement")
print(f"   AOV Uplift: +{(v2['AOV_Uplift']/rb['AOV_Uplift'] - 1)*100:.0f}% more revenue per add-on click")
print(f"   KPT Impact: {v2['KPT_Impact']:.1f}min vs {rb['KPT_Impact']:.1f}min (near-zero delay)")
print(f"   Diversity: {v2['Diversity']:.1f} vs {rb['Diversity']:.1f} unique categories shown")
print(f"\n💰 Monthly incremental revenue @ 500K orders/day: ₹{incremental_monthly/1e7:.1f}Cr")